In [ ]:
import os
import sys
import json
import time
import logging
from xgboost import XGBRegressor
from src.trading.live_trader import LiveTrader

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)

# Подгружаем мид модель
mid_model = XGBRegressor()
mid_model._estimator_type = "regressor"
mid_model.load_model("models/model_mid.json")

# Подгружаем спред модель
spread_model = XGBRegressor()
spread_model._estimator_type = "regressor"
spread_model.load_model("models/model_spread.json")

# Подгружаем параметры торговли
with open('opt_results.json', 'r', encoding='utf-8') as f:
    opt_results = json.load(f)

# Установите API_KEY и API_SECRET в окружении или в .env
trader = LiveTrader(
    mid_model=mid_model,
    spread_model=spread_model,
    api_key=os.getenv("API_KEY"),
    api_secret=os.getenv("API_SECRET"),
    leverage=10,
    alpha=opt_results['alpha'],
    beta=opt_results['beta'],
    gamma=opt_results['gamma'],
    epsilon=opt_results['epsilon'],
    max_position=opt_results['max_position'],
    position_qty=opt_results['position_qty'],
    log_dir=f"./runs/live_{time.time()}",
)

# В Jupyter: await trader.run(...). В скрипте: asyncio.run(trader.run(...))
# grid_resolution_ms: 5000 = 5 с, 100 = 100 ms
await trader.run(symbol="BTCUSDC", grid_resolution_ms=5000)